# W9-D7 概念实验：Virtual CTO ADR Health Check

配套阅读：同名 `.md`。这里不复述阅读材料，而是用小规模、可运行的模型检验其中的架构约束。

## 实验问题

**问题 1：ADR 健康度能否从“文档存在”升级为“决策—实现—验证”三维检查？**

建立小型 ADR 台账：每条原则分别记录设计状态、实现状态和自动化验证状态。

In [ ]:
from dataclasses import dataclass, field, replace
from hashlib import sha256
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
rng = np.random.default_rng(202608)
def canonical_digest(payload):
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return "sha256:" + sha256(canonical.encode()).hexdigest()
@dataclass(frozen=True)
class ADRCheck:
    principle: str
    design: bool
    implemented: bool
    verified: bool

checks = [
    ADRCheck("唯一 SkillRelease 入口", True, True, True),
    ADRCheck("Blueprint 不可变", True, True, True),
    ADRCheck("ReleaseChannel 仅晋升", True, True, False),
    ADRCheck("DeploymentRevision 完整闭包", True, False, False),
    ADRCheck("DigitalEmployeeDefinition 不拥有 Runtime", True, False, False),
]
for c in checks: print(c.principle, "设计/实现/验证 =", c.design, c.implemented, c.verified)
print("设计已覆盖：", sum(c.design for c in checks), "/", len(checks))
print("已验证闭环：", sum(c.verified for c in checks), "/", len(checks))

## 实验问题

**问题 2：只看平均分会掩盖什么？**

把本周五维评分转成可计算向量，比较简单平均与最低维度；CTO 风险优先看短板。

In [ ]:
dimensions = ["架构质量", "代码健康", "ADR 一致性", "技术债", "开发体验"]
scores = np.array([7.5, 6.5, 7.0, 6.0, 7.0])
print("平均：", scores.mean().round(2), "最低维度：", dimensions[scores.argmin()], scores.min())
fig, ax = plt.subplots(figsize=(7.4, 3.6))
ax.bar(dimensions, scores, color=["#1b9e77" if x >= 7 else "#d95f02" for x in scores])
ax.axhline(7, color="gray", ls="--", lw=1, label="健康参考线 7")
ax.set_ylim(0, 10); ax.set_ylabel("评分（10分）"); ax.set_title("健康检查需暴露短板，而非只汇报平均分")
ax.tick_params(axis="x", rotation=20); ax.legend(); plt.tight_layout(); plt.show()

## 实验问题

**问题 3：实现缺口如何转成优先级，而不是停留在感受？**

按风险 = 影响面 × 未实现程度 × 变更频率，为三个目标态对象排序。

In [ ]:
gaps = [
    ("DeploymentRevision 闭包", 10, .9, 8),
    ("TrafficPolicy", 8, .8, 30),
    ("数字员工定义分层", 6, .7, 2),
    ("Blueprint 编译器实质化", 8, .5, 4),
]
prioritized = sorted(((name, impact * missing * frequency) for name, impact, missing, frequency in gaps), key=lambda x: x[1], reverse=True)
for rank, (name, risk) in enumerate(prioritized, 1): print(f"P{rank}: {name}，风险分 {risk:.1f}")
assert prioritized[0][0] == "TrafficPolicy"

## 实验问题

**问题 4：ADR 建议是否应随着证据更新？**

模拟一个健康门：只有“设计已接受 + 实现存在 + 验证通过”的 ADR 才标记为 verified；其余给出下一步。

In [ ]:
def status(c):
    if c.verified: return "verified"
    if c.implemented: return "implemented，补自动验证"
    if c.design: return "accepted，进入实现计划"
    return "draft"
report = {c.principle: status(c) for c in checks}
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report["唯一 SkillRelease 入口"] == "verified"
print("结论：健康检查的输出应是可执行的下一步，而不是静态评分。")